# Homework 8.1 - Task 4

## Reverse Probability-Flow Dynamics and Sign Check

The forward probability-flow ODE is

$$
\frac{dX_t}{dt}
=
f(X_t,t)
-
\frac{1}{2}g(t)^2s(X_t,t).
$$

For the chosen variance-preserving schedule,

$$
f(x,t)
=
-\frac{1}{2}\beta(t)x
$$

and

$$
g(t)^2=\beta(t),
$$

where

$$
\beta(t)
=
\pi
\tan\left(
\frac{\pi t}{2}
\right).
$$

The individual coefficients diverge at $t=1$, so direct numerical evaluation of the unsimplified formula is a bad endpoint implementation.

## 1. Analytic cancellation

Let

$$
\theta
=
\frac{\pi t}{2},
$$

$$
\alpha=\cos(\theta),
$$

and

$$
v=1-q\alpha^2.
$$

The exact score is

$$
s(x,t)
=
\frac{
A\alpha
\tanh\left(
\frac{A\alpha x}{v}
\right)
-x
}{v}.
$$

After substituting the score and simplifying analytically, the forward probability-flow velocity becomes

$$
V(x,t)
=
\frac{
\pi\sin(\theta)
}{
2v
}
\left[
q\alpha x
-
A
\tanh\left(
\frac{A\alpha x}{v}
\right)
\right].
$$

This expression has the continuous endpoint limit

$$
V(x,1)=0.
$$

Therefore the density and the stabilized probability-flow velocity remain regular even though $b(t)$ and $g(t)^2$ diverge separately.

## 2. Increasing reverse clock

We use

$$
\tau=1-t.
$$

Then

$$
\frac{dt}{d\tau}=-1.
$$

Therefore

$$
\frac{dX}{d\tau}
=
-V(X,1-\tau).
$$

The reverse-clock drift is

$$
U(x,t)
=
\frac{
\pi\sin(\theta)
}{
2v
}
\left[
A
\tanh\left(
\frac{A\alpha x}{v}
\right)
-
q\alpha x
\right].
$$

Numerical integration starts from the exact final distribution

$$
X_{\tau=0}\sim N(0,1)
$$

and advances to $\tau=1$, which corresponds to $t=0$.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw8_1"
RESULT_DIR = PROJECT_ROOT / "result images" / "hw8_1"

config = json.loads(
    (
        DATA_DIR
        / "task1_configuration.json"
    ).read_text(
        encoding="utf-8"
    )
)

A = float(config["a"])
SIGMA0 = float(config["sigma0"])
Q = float(config["q"])

In [2]:
def alpha(t):
    return np.cos(
        0.5
        * np.pi
        * np.asarray(
            t,
            dtype=float,
        )
    )


def variance(t):
    a = alpha(t)
    return (
        1.0
        - Q * a**2
    )


def reverse_clock_drift(
    x,
    tau,
    mode_amplitude=A,
):
    t = 1.0 - tau

    theta = (
        0.5
        * np.pi
        * t
    )

    a_t = np.cos(
        theta
    )

    v_t = (
        1.0
        - Q * a_t**2
    )

    sin_theta = np.sin(
        theta
    )

    argument = (
        mode_amplitude
        * a_t
        * x
        / v_t
    )

    return (
        np.pi
        * sin_theta
        / (2.0 * v_t)
        * (
            mode_amplitude
            * np.tanh(
                argument
            )
            - Q
            * a_t
            * x
        )
    )

## 3. Centered-Gaussian sign audit

Set the mode amplitude to zero.

Then the family is the centered Gaussian

$$
N(0,\sigma(t)^2).
$$

During reverse generation, the variance must shrink from 1 to $\sigma_0^2$.

Therefore for $x>0$ the reverse-clock drift should point left, and for $x<0$ it should point right.

This provides an independent sign check.

In [3]:
audit_times = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
]

rows = []

for tau in audit_times:
    drift_positive = float(
        reverse_clock_drift(
            np.array([1.0]),
            tau,
            mode_amplitude=0.0,
        )[0]
    )

    drift_negative = float(
        reverse_clock_drift(
            np.array([-1.0]),
            tau,
            mode_amplitude=0.0,
        )[0]
    )

    rows.append({
        "tau": tau,
        "t": 1.0 - tau,
        "drift_at_x_plus_1": drift_positive,
        "drift_at_x_minus_1": drift_negative,
        "positive_x_contracts": (
            drift_positive < 0
        ),
        "negative_x_contracts": (
            drift_negative > 0
        ),
    })


sign_audit = pd.DataFrame(
    rows
)

sign_audit_path = (
    DATA_DIR
    / "task4_centered_gaussian_sign_audit.csv"
)

sign_audit.to_csv(
    sign_audit_path,
    index=False,
)

print(
    sign_audit.to_string(
        index=False
    )
)

 tau    t  drift_at_x_plus_1  drift_at_x_minus_1  positive_x_contracts  negative_x_contracts
0.05 0.95          -0.115853            0.115853                  True                  True
0.25 0.75          -0.603508            0.603508                  True                  True
0.50 0.50          -1.385997            1.385997                  True                  True
0.75 0.25          -2.605940            2.605940                  True                  True
0.95 0.05          -1.687162            1.687162                  True                  True


## 4. RK4 reverse integrator

The stabilized reverse-clock ODE is integrated using a fixed-step fourth-order Runge-Kutta method.

The centered Gaussian case has an analytic solution.

If the final sample is $Z$, then at $t=0$ the exact reverse result is

$$
X_0=\sigma_0 Z.
$$

We compare the numerical mapping with this exact result.

In [4]:
def rk4_reverse(
    initial_samples,
    n_steps,
    mode_amplitude=A,
):
    x = np.asarray(
        initial_samples,
        dtype=float,
    ).copy()

    h = 1.0 / n_steps
    tau = 0.0

    for _ in range(
        n_steps
    ):
        k1 = reverse_clock_drift(
            x,
            tau,
            mode_amplitude,
        )

        k2 = reverse_clock_drift(
            x
            + 0.5 * h * k1,
            tau + 0.5 * h,
            mode_amplitude,
        )

        k3 = reverse_clock_drift(
            x
            + 0.5 * h * k2,
            tau + 0.5 * h,
            mode_amplitude,
        )

        k4 = reverse_clock_drift(
            x
            + h * k3,
            tau + h,
            mode_amplitude,
        )

        x = (
            x
            + h
            / 6.0
            * (
                k1
                + 2.0 * k2
                + 2.0 * k3
                + k4
            )
        )

        tau += h

    return x


rng = np.random.default_rng(
    63881
)

z = rng.normal(
    size=5000
)

centered_generated = rk4_reverse(
    z,
    n_steps=1000,
    mode_amplitude=0.0,
)

centered_exact = (
    SIGMA0 * z
)

error = (
    centered_generated
    - centered_exact
)

centered_validation = {
    "samples": int(
        len(z)
    ),
    "rk4_steps": 1000,
    "rmse": float(
        np.sqrt(
            np.mean(
                error**2
            )
        )
    ),
    "max_abs_error": float(
        np.max(
            np.abs(
                error
            )
        )
    ),
    "generated_variance": float(
        np.var(
            centered_generated,
            ddof=1,
        )
    ),
    "target_variance": float(
        SIGMA0**2
    ),
}

centered_path = (
    DATA_DIR
    / "task4_centered_gaussian_reverse_validation.json"
)

centered_path.write_text(
    json.dumps(
        centered_validation,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        centered_validation,
        indent=2,
    )
)

{
  "samples": 5000,
  "rk4_steps": 1000,
  "rmse": 6.56808551737408e-13,
  "max_abs_error": 2.2435386881625163e-12,
  "generated_variance": 0.06159761845998677,
  "target_variance": 0.0625
}


# Task 4 Conclusion

The reverse integration uses an increasing reverse clock.

The sign audit confirms contraction in the centered-Gaussian case, and the RK4 implementation is checked against the exact centered-Gaussian map.

Task 5 applies the same validated reverse flow to the two-mode target and compares the generated distribution with the exact target.